# Línea base TF-IDF + regresión logística (ticket #29)

Descarga LIAR, FakeNewsNet y FakeDeS, congela las particiones y entrena la línea base
sobre FakeDeS, sobre el conjunto completo y sobre FakeDeS + afirmaciones de Chequeado (#36;
la C de esta última se elige en la validación de Chequeado, como la última etapa de `beto-chequeado`). Deja un JSON de resultados por corrida en
`resultados/` y el modelo exportado en `modelos/`.

En Colab: *Entorno de ejecución → Ejecutar todas*. No usa GPU. Al final descarga un
`.zip` con los resultados y los modelos, para versionar los JSON y subir el modelo al Space.

In [ ]:
RAMA = "main"   # rama del repo de la que se toma el módulo
SEMILLA = 42
GRILLA_C = [0.1, 1.0, 10.0]   # se elige por F1 macro en validación
ENTRENAR_EN = ["fakedes-chequeado"]   # todas: ["fakedes", "completo", "fakedes-chequeado"]

In [ ]:
import sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # Colab ya trae pandas, scikit-learn, joblib y openpyxl: no se reinstalan para no
    # forzar un reinicio. Las versiones usadas quedan en el JSON (`entorno`).
    !git clone -q --depth 1 -b {RAMA} https://github.com/martinnbejarano/PFI-Wiki.git
    %cd PFI-Wiki/prototipo/clasificador

In [ ]:
import hashlib
import os

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

from datos import AFIRMACIONES_CHEQUEADO, cargar_chequeado, cargar_todos, congelar, descargar, preparar, preprocesar
from resultados import construir_resultado, guardar, metricas

corpus = preparar(cargar_todos(descargar("datos")), SEMILLA)
# Si `particiones.csv.gz` ya está versionado, falla cuando no coincide.
HASH_PARTICIONES = congelar(corpus, "particiones.csv.gz")
print("hash de las particiones:", HASH_PARTICIONES)
chequeado = cargar_chequeado()   # partición propia, sin prueba: no entra en `corpus`
corpus.groupby(["dataset", "particion"])["etiqueta"].value_counts().unstack()

In [ ]:
HIPERPARAMETROS = {
    "tfidf": {"ngram_range": [1, 2], "min_df": 2, "sublinear_tf": True},
    "regresion_logistica": {"class_weight": "balanced", "max_iter": 2000},
}


def construir(C):
    # El preprocesamiento va dentro del modelo: el Space recibe el texto crudo.
    # Para cargar el .joblib hace falta que `datos.preprocesar` sea importable.
    tfidf = dict(HIPERPARAMETROS["tfidf"], ngram_range=tuple(HIPERPARAMETROS["tfidf"]["ngram_range"]))
    return make_pipeline(
        TfidfVectorizer(preprocessor=preprocesar, **tfidf),
        LogisticRegression(C=C, random_state=SEMILLA, **HIPERPARAMETROS["regresion_logistica"]),
    )


def evaluar(modelo, datos):
    return metricas(datos["etiqueta"].tolist(), modelo.predict_proba(datos["texto"])[:, 1].tolist())


def particion(datos, nombre):
    return datos[datos["particion"] == nombre]

In [ ]:
os.makedirs("modelos", exist_ok=True)
fakedes = corpus[corpus["dataset"] == "fakedes"]
# entrenamiento, validación
CONJUNTOS = {
    "fakedes": (particion(fakedes, "entrenamiento"), particion(fakedes, "validacion")),
    "completo": (particion(corpus, "entrenamiento"), particion(corpus, "validacion")),
    "fakedes-chequeado": (pd.concat([particion(fakedes, "entrenamiento"), particion(chequeado, "entrenamiento")]), particion(chequeado, "validacion")),
}
tabla = []

for entrenado_en in ENTRENAR_EN:
    entrenamiento, validacion = CONJUNTOS[entrenado_en]
    f1_por_C = {}
    for C in GRILLA_C:
        candidato = construir(C).fit(entrenamiento["texto"], entrenamiento["etiqueta"] == "falso")
        f1_por_C[C] = evaluar(candidato, validacion)["f1_macro"]
        if f1_por_C[C] == max(f1_por_C.values()):
            modelo, mejor_C = candidato, C

    evaluaciones = {f"{entrenado_en.removeprefix('fakedes-')}/validacion": evaluar(modelo, validacion)}
    for nombre in {"completo": ["completo", "fakedes", "liar", "fakenewsnet"], "fakedes-chequeado": ["fakedes"]}.get(entrenado_en, [entrenado_en]):
        datos_prueba = corpus if nombre == "completo" else corpus[corpus["dataset"] == nombre]
        evaluaciones[f"{nombre}/prueba"] = evaluar(modelo, particion(datos_prueba, "prueba"))

    id_corrida = f"tfidf-lr_{entrenado_en}_s{SEMILLA}"
    resultado = construir_resultado(
        id_corrida=id_corrida,
        modelo="tfidf-lr",
        entrenado_en=entrenado_en,
        hiperparametros=dict(HIPERPARAMETROS, C=mejor_C, f1_macro_validacion_por_C=f1_por_C,
                             **({"hash_chequeado": hashlib.sha256(AFIRMACIONES_CHEQUEADO.read_bytes()).hexdigest()} if "chequeado" in entrenado_en else {})),
        evaluaciones=evaluaciones,
        semilla=SEMILLA,
        hash_particiones=HASH_PARTICIONES,
    )
    print(guardar(resultado, "resultados"))
    joblib.dump(modelo, f"modelos/{id_corrida}.joblib")
    for evaluacion, m in evaluaciones.items():
        tabla.append({"corrida": id_corrida, "evaluacion": evaluacion, "f1_macro": m["f1_macro"], "auc_roc": m["auc_roc"], "n": m["n"]})

pd.DataFrame(tabla).round(4)

In [ ]:
if EN_COLAB:
    from google.colab import files
    !zip -q -r linea_base.zip resultados modelos particiones.csv.gz
    files.download("linea_base.zip")